In [7]:
import pandas as pd
import getpass
from huggingface_hub import notebook_login
import os

from datasets import load_dataset
from fireworks.client import Fireworks
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from openai import OpenAI

In [8]:
notebook_login()

In [9]:
os.environ["FIREWORKS_API_KEY"] = getpass.getpass("fireworks api:")
client = Fireworks(api_key=os.environ["FIREWORKS_API_KEY"])

In [10]:
# sample_data = pd.read_excel('english-danish-openai.xlsx')

# sample_data.to_csv ("english-danish-openai.csv",
#                   index = None,
#                   header=True)

english_danish_sample_data = pd.read_csv('english-danish-openai.csv')
english_danish_sample_data

,English,Danish
0,The cat sat on the windowsill watching the bi...,Katten sad på vindueskarmen og kiggede på fugl...
1,She baked a delicious chocolate cake for her f...,Hun bagte en lækker chokoladekage til sin vens...
2,The new park in the city center has become a p...,Den nye park i byens centrum er blevet et popu...
3,He quickly realized that learning a new langua...,"Han indså hurtigt, at det at lære et nyt sprog..."
4,The old castle on the hill offers a beautiful ...,Det gamle slot på bakken tilbyder en smuk udsi...
...,...,...
114,He enjoyed listening to classical music while ...,"Han nød at lytte til klassisk musik, mens han ..."
115,The old library was filled with countless trea...,Det gamle bibliotek var fyldt med utallige ska...
116,She spent her afternoons tending to her garden.,Hun tilbragte sine eftermiddage med at passe s...
117,The children's laughter echoed in the playground.,Børnenes latter genlød på legepladsen.


In [14]:
instruction = """
          you are an expert translator between English and Danish

          #user will provide you sentences in English 
          #translate users english sentence to Danish.
          #you can only use users english sentence
          
  """

def translate_english_to_danish(english_samples_csv_file, model):
    list_of_danish_sentences = list()
    df = pd.read_csv(english_samples_csv_file)
    for i, row in enumerate(df.iterrows()):
        danish_sentence = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": instruction},
              {"role": "user", "content": row[1]['English']}
            ],
        )
        response = danish_sentence.choices[0].message.content
        list_of_danish_sentences.append(response)    
    return list_of_danish_sentences

    


In [15]:
llama_8b_translation = translate_english_to_danish('./english-danish-openai.csv','accounts/fireworks/models/llama-v3-8b-instruct')



In [16]:
# english_danish_sample_data["Danish_llama_3"] = llama_8b_translation

llama_8b_translation[0]

'Katten sad på vindueskarmen og betragtede fuglene udenfor.'

In [17]:
english_danish_sample_data

,English,Danish
0,The cat sat on the windowsill watching the bi...,Katten sad på vindueskarmen og kiggede på fugl...
1,She baked a delicious chocolate cake for her f...,Hun bagte en lækker chokoladekage til sin vens...
2,The new park in the city center has become a p...,Den nye park i byens centrum er blevet et popu...
3,He quickly realized that learning a new langua...,"Han indså hurtigt, at det at lære et nyt sprog..."
4,The old castle on the hill offers a beautiful ...,Det gamle slot på bakken tilbyder en smuk udsi...
...,...,...
114,He enjoyed listening to classical music while ...,"Han nød at lytte til klassisk musik, mens han ..."
115,The old library was filled with countless trea...,Det gamle bibliotek var fyldt med utallige ska...
116,She spent her afternoons tending to her garden.,Hun tilbragte sine eftermiddage med at passe s...
117,The children's laughter echoed in the playground.,Børnenes latter genlød på legepladsen.


In [18]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("openai api:")


openai_client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [19]:
# instruction = """
#         you are an expert translator between English and Danish

#         #user will only give you samples of a sentence translated from English to Danish#
#         # Give the translation a score on a scale from one to ten#
#         #format should be something like 'score 1'
#         #Think through your reasoning step-by-step and write the score at the end.#
          
#   """


instruction = """
        you are an expert translator between English and Danish

        #user will only give you samples of a sentence translated from English to Danish#
        # Give the translation a score on a scale from one to ten#
        #format should only be your score          
  """


def evaluate_danish_sentences(english_danish_open_csv, evaluation_sentences):
    scores = list()
    df = pd.read_csv(english_danish_open_csv)
    df["Danish_llama_3"] = evaluation_sentences

    for i, row in enumerate(df.iterrows()): 
        
        response = openai_client.chat.completions.create(
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": f'''English:{row[1]['English']} 
                                                Danish:{row[1]['Danish_llama_3']}'''}
            ],
            model="gpt-4o",

        )
        try:
            response = response.choices[0].message.content
            #score = int(json.loads(response.split('\n')[-1])['Score'])  
            scores.append(int(response))
        except json.JSONDecodeError as jde:
            continue

    return sum(scores) / len(scores)


llama_8b_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", llama_8b_translation)

In [20]:
f'''For the base model score is {round(llama_8b_avg_score,2)}'''

'For the base model score is 5.98'

In [21]:
!firectl list models

NAME                              CREATE TIME          KIND           CHAT  PUBLIC  STATE  STATUS MESSAGE
755fd9d02a7046ef8368d4a15c86f3cc  2024-07-05 20:10:19  HF_PEFT_ADDON  true  false   READY  
83ba6bb9b2ec4ca48bad7e473b87a113  2024-07-05 21:33:24  HF_PEFT_ADDON  true  false   READY  
bfa866cd4b4841669796a0deee36d771  2024-06-27 21:36:41  HF_PEFT_ADDON  true  false   READY  
c2c7013101774cd19f0a18cc2f109a29  2024-06-27 21:08:19  HF_PEFT_ADDON  true  false   READY  

Total size: 4


In [22]:
dataset_id = 'danish-to-english-dataset-v1'
!firectl get dataset {dataset_id}

Name: accounts/m44rkt1-d483d1/datasets/danish-to-english-dataset-v1
Display Name: 
Create Time: 2024-07-05 19:45:11
State: READY
Status: OK
Example Count: 119


In [38]:
ft_english_danish_model = '83ba6bb9b2ec4ca48bad7e473b87a113'
full_path_english_danish_model = f'accounts/m44rkt1-d483d1/models/{ft_english_danish_model}'

!firectl get model {ft_english_danish_model}

Name: accounts/m44rkt1-d483d1/models/83ba6bb9b2ec4ca48bad7e473b87a113
Display Name: 
Description: 
Create Time: 2024-07-05 21:33:24
Created By: 
State: READY
Status: OK
Kind: HF_PEFT_ADDON
Deployment Id: 
Github Url: 
Hugging Face Url: 
Peft Details:
  Base Model: accounts/fireworks/models/llama-v3-8b-instruct-hf
  R: 32
  Target Modules: [
    down_proj, 
    o_proj, 
    v_proj, 
    q_proj, 
    gate_proj, 
    k_proj, 
    up_proj
  ]
Public: false
Conversation Config:
  Style: jinja
  System: 
  Template: 
Context Length: 8192
Supports Image Input: false
Supports Tools: false
Imported From: 
Fine Tuning Job: accounts/m44rkt1-d483d1/fineTuningJobs/83ba6bb9b2ec4ca48bad7e473b87a113
Default Draft Model: 
Default Draft Token Count: 0
Precisions: []
Deployed Model Refs: []


In [46]:
ft_model_name = f'accounts/m44rkt1-d483d1/models/7e4933e113ae4c1daf02f9ed14c4e416'
base_model_name = "accounts/fireworks/models/llama-v3-8b-instruct"

sample_data = pd.read_csv('english-danish-openai.csv')

sample_data

,English,Danish
0,The cat sat on the windowsill watching the bi...,Katten sad på vindueskarmen og kiggede på fugl...
1,She baked a delicious chocolate cake for her f...,Hun bagte en lækker chokoladekage til sin vens...
2,The new park in the city center has become a p...,Den nye park i byens centrum er blevet et popu...
3,He quickly realized that learning a new langua...,"Han indså hurtigt, at det at lære et nyt sprog..."
4,The old castle on the hill offers a beautiful ...,Det gamle slot på bakken tilbyder en smuk udsi...
...,...,...
114,He enjoyed listening to classical music while ...,"Han nød at lytte til klassisk musik, mens han ..."
115,The old library was filled with countless trea...,Det gamle bibliotek var fyldt med utallige ska...
116,She spent her afternoons tending to her garden.,Hun tilbragte sine eftermiddage med at passe s...
117,The children's laughter echoed in the playground.,Børnenes latter genlød på legepladsen.


In [48]:

def generate_translations(model, english_sentences):
    responses = list()
    for i, sentence in enumerate(english_sentences):
        msg = [
            {"role": "system", "content": 'Translate the English sentence to Danish. Your response must contain ONLY the translated sentence.'},
            {"role": "user", "content": sentence},
        ]
        response = client.chat.completions.create(
            model=model,
            messages=msg,
            temperature=0,
        )

        response = response.choices[0].message.content
        print(response)
        responses.append(response)
    return responses

finetuned_generated_danish = generate_translations(ft_model_name , sample_data['English'].tolist())

finetuned_generated_danish

<|start_header_id|>Katzen sat på vindueskarmen og kiggede på fuglene udenfor.
<|start_header_id|> Hun bagte en lækker chokoladekage til sin vens fødsel.
<|start_header_id|>Den nye park i byens centrum er blevet et populært sted for familier.
<|start_header_id|><|start_header_id|>Han indførte hurtigt, at lære et nyt sprog kræver tålmodighed og praksis.
<|start_header_id|>```
<|start_header_id|>De besluttede at tilbringe deres ferie med at udforske de fjerne øer i Stillehavet.
<|start_header_id|>"Forskeren gjorde en banebrydende opdagelse, der kunne ændre medicinens fremtid."
<|start_header_id|><|start_header_id|>Børnene var begejstrede for at starte deres første skoledag efter somferien.
<|start_header_id|><|start_header_id|>Hun gennemførte maratonen trods de udfordrende vejrforhold.
<|start_header_id|>Den innovative startup har til formål at skabe bæredygtige løsninger til byliv.
<|start_header_id|> Hans bedstefar fortalte ham historier om de gamle dage, da landsbyen var meget mindre.


['<|start_header_id|>Katzen sat på vindueskarmen og kiggede på fuglene udenfor.',
 '<|start_header_id|> Hun bagte en lækker chokoladekage til sin vens fødsel.',
 '<|start_header_id|>Den nye park i byens centrum er blevet et populært sted for familier.',
 '<|start_header_id|><|start_header_id|>Han indførte hurtigt, at lære et nyt sprog kræver tålmodighed og praksis.',
 '<|start_header_id|>```',
 '<|start_header_id|>De besluttede at tilbringe deres ferie med at udforske de fjerne øer i Stillehavet.',
 '<|start_header_id|>"Forskeren gjorde en banebrydende opdagelse, der kunne ændre medicinens fremtid."',
 '<|start_header_id|><|start_header_id|>Børnene var begejstrede for at starte deres første skoledag efter somferien.',
 '<|start_header_id|><|start_header_id|>Hun gennemførte maratonen trods de udfordrende vejrforhold.',
 '<|start_header_id|>Den innovative startup har til formål at skabe bæredygtige løsninger til byliv.',
 '<|start_header_id|> Hans bedstefar fortalte ham historier om de g

In [49]:
finetuned_generated_danish_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", finetuned_generated_danish)

finetuned_generated_danish_avg_score

7.033613445378151

In [51]:
f'''For the finetuned model score is {round(finetuned_generated_danish_avg_score,2)}'''

'For the finetuned model score is 7.03'